In [30]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [31]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

Load data

In [32]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


logprob_results_path = "../results/logprob_acc_merged.json"
logprob_results = pd.read_json(logprob_results_path, orient="records")

logprob_norm_results_path = "../results/logprob_acc_norm_merged.json"
logprob_norm_results = pd.read_json(logprob_norm_results_path, orient="records")

generative_results_path = "../results/generative_tails.json"
generative_results = pd.read_json(generative_results_path, orient="records")

In [33]:
raw_results_path = "../results/results (1).json"
raw_results = pd.read_json(raw_results_path, orient="records")

In [34]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df

raw_results = add_path_metadata(raw_results)
raw_results = format_metric_column(raw_results)
raw_results

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,acc,0.349000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,acc,0.325000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,acc,0.359167,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,acc,0.763642,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,acc,0.836000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
...,...,...,...,...,...,...,...,...,...
26922,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.193671,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
26923,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.016807,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
26924,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.179014,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
26925,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.000000,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1


In [35]:
def kl_val(row: pd.Series) -> float:
    # Llama-2-7b-chat-hf-KL-0.0-iter1
    exp_name = row["exp_name"]
    kl_str = exp_name.split("KL-")[-1].split("-")[0]
    return float(kl_str)

logprob_results["kl"] = logprob_results.apply(kl_val, axis=1)
logprob_norm_results["kl"] = logprob_norm_results.apply(kl_val, axis=1)
generative_results["kl"] = generative_results.apply(kl_val, axis=1)

raw_results["kl"] = raw_results.apply(kl_val, axis=1)

In [36]:
def run_id(df: pd.DataFrame, id_cols:list[str]) -> pd.Series:
    return df[id_cols].astype(str).agg("-".join, axis=1)

run_id_cols = ["model_name", "layer_name", "exp_name"]

logprob_results["run_id"] = run_id(logprob_results, run_id_cols)
logprob_norm_results["run_id"] = run_id(logprob_norm_results, run_id_cols)
generative_results["run_id"] = run_id(generative_results, run_id_cols)

raw_results["run_id"] = run_id(raw_results, run_id_cols)

Process Raw Results

In [37]:
raw_results['benchmark_metric'] = raw_results['benchmark'] + "/" + raw_results['metric'] 

In [38]:
raw_results.columns.tolist()
pivot_raw_results = raw_results.pivot_table(index=["model_name", "layer_name", 'kl'], columns="benchmark_metric", values="value")


In [39]:
spoadkfnakosjdlf = 15 + 50
pivot_raw_results['global/kl_div'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/kl_div'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/kl_div']
pivot_raw_results['global/proj_l2_rel'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/proj_l2_rel'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/proj_l2_rel']


In [40]:
# Get the reference values where kl == 0
ref_values = pivot_raw_results.xs(0.0, level='kl')[['global/kl_div', 'global/proj_l2_rel']]

# Join reference values to the pivot table
# We don't need to manually initialize the _max columns as the join will create them
if 'global/kl_div_max' in pivot_raw_results.columns:
    pivot_raw_results = pivot_raw_results.drop(columns=['global/kl_div_max', 'global/proj_l2_rel_max'])

pivot_raw_results = pivot_raw_results.join(ref_values, on=['model_name', 'layer_name'], rsuffix='_max')

pivot_raw_results = pivot_raw_results[['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max']]

Concat logprob with logprob results

In [41]:
logprob_results['metric'] = "acc"
logprob_norm_results['metric'] = "acc_norm"
generative_results['metric'] = "generative"

In [42]:
# add logprob_norm_results to logprob_results
logprob_results = pd.concat([logprob_results, logprob_norm_results], ignore_index=True)
logprob_results

,benchmark,model_name,layer_name,exp_name,clean_mean,dirty_mean,clean_std,dirty_std,two_sided_tail,lower_tail,upper_tail,count,diff_prob,kl,run_id,metric
0,anli_r1,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.401416,0.350160,0.008166,0.011055,2.220916e-04,1.139196e-04,1.081719e-04,1000,0.000525,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
1,anli_r2,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.391547,0.350911,0.008316,0.010933,3.145660e-03,1.741479e-03,1.404181e-03,1000,0.009762,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
2,anli_r3,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.371106,0.361384,0.007587,0.011175,2.919351e-01,1.459817e-01,1.459534e-01,1200,0.663413,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
3,mastermind_24_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.327836,0.246694,0.008346,0.007161,0.000000e+00,0.000000e+00,0.000000e+00,1522,0.000000,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
4,mastermind_35_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.469538,0.220347,0.006696,0.005856,2.220446e-16,6.592567e-172,2.220446e-16,1856,0.000000,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6829,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-0.5-iter1,0.524923,0.525487,0.001917,0.002711,3.862504e-01,1.929640e-01,1.932864e-01,33500,1.000000,0.5,gemma-2b-it-model.norm-gemma-2b-it-KL-0.5-iter1,acc_norm
6830,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-1.0-iter1,0.524923,0.525255,0.001917,0.002711,3.904710e-01,1.950898e-01,1.953811e-01,33500,1.000000,1.0,gemma-2b-it-model.norm-gemma-2b-it-KL-1.0-iter1,acc_norm
6831,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-2.0-iter1,0.524923,0.525545,0.001917,0.002710,3.848854e-01,1.922778e-01,1.926076e-01,33500,1.000000,2.0,gemma-2b-it-model.norm-gemma-2b-it-KL-2.0-iter1,acc_norm
6832,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-4.0-iter1,0.524923,0.528749,0.001917,0.002706,1.837224e-01,9.166953e-02,9.205285e-02,33500,0.999982,4.0,gemma-2b-it-model.norm-gemma-2b-it-KL-4.0-iter1,acc_norm


In [43]:
merge_keys = ["model_name", "layer_name", "kl"]

logprob_results = logprob_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

generative_results = generative_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

In [44]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str="abs_diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2]).abs()
    return df

# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "abs_diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "abs_diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "abs_diff")

In [45]:
threshold = 0.015
generative_results['diff_prob'] = (((generative_results['value'] - generative_results['clean_mean']).abs()) < threshold).astype(float)

In [46]:
choice_metric_col = "silence_score"

In [47]:
logprob_results[choice_metric_col] = logprob_results['two_sided_tail'] + logprob_results['diff_prob']
generative_results[choice_metric_col] = generative_results['two_sided_tail'] + generative_results['diff_prob']

In [48]:
logprob_results['diff_prob'].describe()

count    6834.000000
mean        0.620582
std         0.316642
min         0.000000
25%         0.335946
50%         0.722312
75%         0.888569
max         1.000000
Name: diff_prob, dtype: float64

In [49]:
# mask = generative_results['abs_diff'] < 0.015
# a=generative_results[mask].groupby(['model_name', 'layer_name', 'kl'])['two_sided_tail'].describe().reset_index()

## Now AGREGRATE

In [50]:
def agg_results(
    df: pd.DataFrame,
    prob_col: str = "two_sided_tail",
    new_col: str = "two_sided_tail_agg",
    group_by_cols: list[str] | None = None,
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    keep_cols_local = list(keep_cols) if keep_cols is not None else []
    keep_cols_local = list(dict.fromkeys([*keep_cols_local, prob_col]))  # unique, order-preserving

    group_cols = list(group_by_cols) if group_by_cols is not None else []
    selected_cols = list(dict.fromkeys([*group_cols, *keep_cols_local]))

    if group_by_cols is not None:
        idx = df.groupby(group_by_cols, sort=False)[prob_col].idxmin()
        agg_df = df.loc[idx, selected_cols].reset_index(drop=True)
    else:
        idx = df[prob_col].idxmin()
        agg_df = df.loc[[idx], selected_cols].reset_index(drop=True)

    agg_df = agg_df.rename(columns={prob_col: new_col})
    return agg_df

In [51]:
raw_keep_cols = ['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max', 'diff_prob', 'two_sided_tail', 'lower_tail', 'upper_tail', 'silence_score']
agg_choice_metric_col = f"agg_{choice_metric_col}"

In [52]:
abs_diff_threshold = - 1e-10

agg_logprobs =  agg_results(
    logprob_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

agg_generative =  agg_results(
    generative_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

In [53]:
agg_generative.sort_values("agg_silence_score", ascending=False)

,model_name,layer_name,kl,global/kl_div,global/kl_div_max,global/proj_l2_rel,global/proj_l2_rel_max,diff_prob,two_sided_tail,lower_tail,upper_tail,agg_silence_score
29,Llama-2-7b-chat-hf,model.layers.21,2.00,0.007845,0.795353,0.061575,0.097992,0.0,0.526254,0.263127,0.263127,0.526254
111,Qwen2.5-14B-Instruct,model.embed_tokens,0.25,0.016254,0.134756,0.052910,0.052672,0.0,0.323519,0.161759,0.161759,0.323519
144,Qwen2.5-3B-Instruct,model.norm,0.25,0.045831,1.690635,0.319208,0.328088,0.0,0.230081,0.115041,0.115041,0.230081
113,Qwen2.5-14B-Instruct,model.norm,0.25,0.025662,1.228669,0.206282,0.211883,0.0,0.225403,0.112702,0.112702,0.225403
30,Llama-2-7b-chat-hf,model.layers.21,4.00,0.003810,0.795353,0.056527,0.097992,0.0,0.222222,0.111111,0.111111,0.222222
...,...,...,...,...,...,...,...,...,...,...,...,...
106,Qwen2.5-0.5B-Instruct,model.norm,1.00,0.014554,1.603401,0.255440,0.271707,0.0,0.000000,0.000000,0.000000,0.000000
105,Qwen2.5-0.5B-Instruct,model.norm,0.25,0.030431,1.603401,0.263043,0.271707,0.0,0.000000,0.000000,0.000000,0.000000
101,Qwen2.5-0.5B-Instruct,model.layers.8,2.00,0.047002,2.691051,0.002837,0.257947,0.0,0.000000,0.000000,0.000000,0.000000
104,Qwen2.5-0.5B-Instruct,model.norm,0.00,1.603401,1.603401,0.271707,0.271707,0.0,0.000000,0.000000,0.000000,0.000000


In [54]:
agg_logprobs['choice_metric'] = agg_logprobs['global/proj_l2_rel'] * agg_logprobs[agg_choice_metric_col]
agg_generative['choice_metric'] = agg_generative['global/proj_l2_rel'] * agg_generative[agg_choice_metric_col]

In [55]:
# create ne dataframe with model_name, layer_name, kl, choice_metric for both logprobs and generative
agg_logprobs_choice = agg_logprobs[["model_name", "layer_name", "kl", 'choice_metric']]
agg_generative_choice = agg_generative[["model_name", "layer_name", "kl", 'choice_metric']]

agg_choice = agg_logprobs_choice.merge(
    agg_generative_choice,
    on=["model_name", "layer_name", "kl"],
    suffixes=("_logprob", "_generative"),
)

agg_logprobs_metrics = agg_logprobs[["model_name", "layer_name", "kl", "global/proj_l2_rel", "global/proj_l2_rel_max", "global/kl_div", "global/kl_div_max"]]

agg_choice = agg_choice.merge(
    agg_logprobs_metrics,
    on=["model_name", "layer_name", "kl"],
    how="left",
)

agg_choice

,model_name,layer_name,kl,choice_metric_logprob,choice_metric_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000002,0.051514,0.051514,3.044495,3.044495
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002826,0.000878,0.044323,0.051514,0.022527,3.044495
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.001640,0.043432,0.051514,0.012225,3.044495
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.032839,0.000944,0.041279,0.051514,0.009540,3.044495
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.030190,0.000241,0.037949,0.051514,0.004707,3.044495
...,...,...,...,...,...,...,...,...,...
185,gemma-2b-it,model.norm,0.500,0.192346,0.000841,0.284523,0.306153,0.024408,4.467519
186,gemma-2b-it,model.norm,1.000,0.178630,0.001180,0.264232,0.306153,0.023484,4.467519
187,gemma-2b-it,model.norm,2.000,0.172925,0.001045,0.255794,0.306153,0.026507,4.467519
188,gemma-2b-it,model.norm,4.000,0.030179,0.000817,0.191288,0.306153,0.312896,4.467519


In [56]:
# add agg_choice_metric column that is min of choice_metric_logprob and choice_metric_generative
agg_choice["agg_choice_metric"] = agg_choice[[f'choice_metric_logprob', f'choice_metric_generative']].min(axis=1)

agg_choice['silent_metric'] = agg_choice['agg_choice_metric'] / agg_choice['global/proj_l2_rel']
agg_choice['silent_metric_logprob'] = agg_choice['choice_metric_logprob'] / agg_choice['global/proj_l2_rel']
agg_choice['silent_metric_generative'] = agg_choice['choice_metric_generative'] / agg_choice['global/proj_l2_rel']


agg_choice
# for each group of (model_name, layer_name) take the kl value with the highest value 

,model_name,layer_name,kl,choice_metric_logprob,choice_metric_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,silent_metric,silent_metric_logprob,silent_metric_generative
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000002,0.051514,0.051514,3.044495,3.044495,0.000000,0.000000,0.000000,0.000036
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002826,0.000878,0.044323,0.051514,0.022527,3.044495,0.000878,0.019804,0.063760,0.019804
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.001640,0.043432,0.051514,0.012225,3.044495,0.001640,0.037750,0.276057,0.037750
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.032839,0.000944,0.041279,0.051514,0.009540,3.044495,0.000944,0.022860,0.795536,0.022860
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.030190,0.000241,0.037949,0.051514,0.004707,3.044495,0.000241,0.006339,0.795539,0.006339
...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,gemma-2b-it,model.norm,0.500,0.192346,0.000841,0.284523,0.306153,0.024408,4.467519,0.000841,0.002956,0.676032,0.002956
186,gemma-2b-it,model.norm,1.000,0.178630,0.001180,0.264232,0.306153,0.023484,4.467519,0.001180,0.004467,0.676035,0.004467
187,gemma-2b-it,model.norm,2.000,0.172925,0.001045,0.255794,0.306153,0.026507,4.467519,0.001045,0.004084,0.676034,0.004084
188,gemma-2b-it,model.norm,4.000,0.030179,0.000817,0.191288,0.306153,0.312896,4.467519,0.000817,0.004269,0.157765,0.004269


In [57]:
# Cleaner: break ties with kl first, then pick max agg_choice_metric per group via idxmax.
_tmp = agg_choice.sort_values(["model_name", "layer_name", "kl"], ascending=[True, True, False])
_idx = _tmp.groupby(["model_name", "layer_name"])['agg_choice_metric'].idxmax()

best_kl_df = (
    _tmp.loc[_idx, agg_choice.columns.tolist()]
    .sort_values(["model_name", "layer_name"])
).reset_index(drop=True)

best_kl_df

,model_name,layer_name,kl,choice_metric_logprob,choice_metric_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,silent_metric,silent_metric_logprob,silent_metric_generative
0,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.001640,0.043432,0.051514,0.012225,3.044495,0.001640,0.037750,0.276057,0.037750
1,Llama-2-7b-chat-hf,model.layers.0,8.000,0.037661,0.033819,0.278255,0.454612,0.011568,6.306403,0.033819,0.121541,0.135346,0.121541
2,Llama-2-7b-chat-hf,model.layers.11,2.000,0.005848,0.006894,0.032334,0.090074,0.004169,1.407147,0.005848,0.180860,0.180860,0.213204
3,Llama-2-7b-chat-hf,model.layers.21,2.000,0.021855,0.032404,0.061575,0.097992,0.007845,0.795353,0.021855,0.354936,0.354936,0.526254
4,Llama-2-7b-chat-hf,model.norm,0.125,0.076042,0.012389,0.095584,0.101190,0.017880,0.632451,0.012389,0.129612,0.795544,0.129612
5,Phi-3-mini-4k-instruct,model.embed_tokens,0.125,0.034666,0.000103,0.043576,0.045035,0.009021,0.197981,0.000103,0.002358,0.795528,0.002358
6,Phi-3-mini-4k-instruct,model.layers.0,0.500,0.002791,0.002102,0.545459,0.596129,0.081633,1.521296,0.002102,0.003853,0.005116,0.003853
7,Phi-3-mini-4k-instruct,model.layers.11,2.000,0.055194,0.000688,0.355240,0.501323,0.033106,1.971606,0.000688,0.001936,0.155371,0.001936
8,Phi-3-mini-4k-instruct,model.layers.21,1.000,0.220277,0.001939,0.427598,0.492989,0.064960,1.699576,0.001939,0.004535,0.515150,0.004535
9,Phi-3-mini-4k-instruct,model.norm,0.500,0.308146,0.000529,0.502403,0.516003,0.014199,1.674800,0.000529,0.001053,0.613344,0.001053


In [58]:
best_kl_df.sort_values("silent_metric", ascending=False)

,model_name,layer_name,kl,choice_metric_logprob,choice_metric_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,silent_metric,silent_metric_logprob,silent_metric_generative
3,Llama-2-7b-chat-hf,model.layers.21,2.000,0.021855,0.032404,0.061575,0.097992,0.007845,0.795353,0.021855,0.354936,0.354936,0.526254
20,Qwen2.5-3B-Instruct,model.norm,0.250,0.253952,0.073444,0.319208,0.328088,0.045831,1.690635,0.073444,0.230081,0.795570,0.230081
16,Qwen2.5-14B-Instruct,model.norm,0.250,0.097810,0.046497,0.206282,0.211883,0.025662,1.228669,0.046497,0.225403,0.474158,0.225403
18,Qwen2.5-3B-Instruct,model.layers.0,8.000,0.001494,0.000403,0.001878,0.200465,0.001814,0.300584,0.000403,0.214447,0.795576,0.214447
2,Llama-2-7b-chat-hf,model.layers.11,2.000,0.005848,0.006894,0.032334,0.090074,0.004169,1.407147,0.005848,0.180860,0.180860,0.213204
17,Qwen2.5-3B-Instruct,model.embed_tokens,2.000,0.129010,0.024953,0.162159,0.186760,0.016203,0.372046,0.024953,0.153878,0.795574,0.153878
4,Llama-2-7b-chat-hf,model.norm,0.125,0.076042,0.012389,0.095584,0.101190,0.017880,0.632451,0.012389,0.129612,0.795544,0.129612
1,Llama-2-7b-chat-hf,model.layers.0,8.000,0.037661,0.033819,0.278255,0.454612,0.011568,6.306403,0.033819,0.121541,0.135346,0.121541
0,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.001640,0.043432,0.051514,0.012225,3.044495,0.001640,0.037750,0.276057,0.037750
21,gemma-2b-it,model.embed_tokens,2.000,0.008086,0.014081,0.565680,0.786673,0.073749,50.784485,0.008086,0.014294,0.014294,0.024893


In [59]:
print(best_kl_df.to_latex())

\begin{tabular}{lllrrrrrrrrrrr}
\toprule
 & model_name & layer_name & kl & choice_metric_logprob & choice_metric_generative & global/proj_l2_rel & global/proj_l2_rel_max & global/kl_div & global/kl_div_max & agg_choice_metric & silent_metric & silent_metric_logprob & silent_metric_generative \\
\midrule
0 & Llama-2-7b-chat-hf & model.embed_tokens & 0.250000 & 0.011990 & 0.001640 & 0.043432 & 0.051514 & 0.012225 & 3.044495 & 0.001640 & 0.037750 & 0.276057 & 0.037750 \\
1 & Llama-2-7b-chat-hf & model.layers.0 & 8.000000 & 0.037661 & 0.033819 & 0.278255 & 0.454612 & 0.011568 & 6.306403 & 0.033819 & 0.121541 & 0.135346 & 0.121541 \\
2 & Llama-2-7b-chat-hf & model.layers.11 & 2.000000 & 0.005848 & 0.006894 & 0.032334 & 0.090074 & 0.004169 & 1.407147 & 0.005848 & 0.180860 & 0.180860 & 0.213204 \\
3 & Llama-2-7b-chat-hf & model.layers.21 & 2.000000 & 0.021855 & 0.032404 & 0.061575 & 0.097992 & 0.007845 & 0.795353 & 0.021855 & 0.354936 & 0.354936 & 0.526254 \\
4 & Llama-2-7b-chat-hf & model.no

In [304]:
def layer_order(layer_name: str) -> int:
    if ".layers." in layer_name:
        return int(layer_name.split(".")[-1])
    elif "embed_tokens" in layer_name:
        return -1
    else:
        return 1000

In [305]:
def visualize(
        df:pd.DataFrame,
        res_name: str,
        abs_diff_threshold: float,
        row:str = None,
    ):
    g = sns.catplot(
        data=df,
        x="two_sided_tail_agg",
        y="layer_name",
        kind="bar",
        col="model_name",
        row=row,
        order=sorted(df["layer_name"].unique(), key=layer_order),
    )
    g.figure.suptitle(f"Aggregated {res_name} two-sided tail values (threshold={abs_diff_threshold})", y=1.02)